## OBIMD TO SA-1B FORMAT

In [ ]:
import json
import cv2
from matplotlib import pyplot as plt
import numpy as np
import os
from PIL import Image
from pycocotools import mask as maskUtils
from tqdm import tqdm


with open('data/label.json') as f:
    datas = json.load(f)
image_root = 'data/OBIMD_rubbing'
output_dir = 'data/OBIMD_facsimile_json'
char_id = 0
for image_id, data in tqdm(enumerate(datas)):
    image_name = os.path.basename(data['Rubbing'])
    facsimile = cv2.imread(os.path.join('data/OBIMD_facsimile_no_boarder', image_name), flags=cv2.IMREAD_GRAYSCALE)
    # CHECK SHAPE etc.
    W, H = Image.open(os.path.join(image_root, image_name)).size
    image_info = dict(image_id=image_id, width=W, height=H, file_name=image_name)
    annotations = []
    oracle_chars = []
    for sentence in data['RecordUtilSentenceGroupVoList']:
        for char in sentence["RecordUtilOracleCharVoList"]:
            x, y, w, h = list(map(int, char['Position'].split(',')))
            try:
                if np.any(facsimile[y:min(y+h, H), x:min(x+w, W)] > 0):
                    oracle_chars.append(char) # get available chars
            except:
                pass
    for char in oracle_chars:
        mask = np.zeros_like(facsimile)
        x, y, w, h = list(map(int, char['Position'].split(',')))
        mask[y:min(y+h, H), x:min(x+w, W)] = facsimile[y:min(y+h, H), x:min(x+w, W)] # TOFIX: check area?
        for _char in oracle_chars:
            if char == _char:
                continue
            _x, _y, _w, _h = list(map(int, _char['Position'].split(',')))
            mask[_y:min(_y+_h, H), _x:min(_x+_w, W)] = 0 # remove other chars
        mask = np.asfortranarray(mask)
        rle = maskUtils.encode(mask)
        rle['counts'] = rle['counts'].decode('utf-8')
        area = maskUtils.area(rle)
        bbox = maskUtils.toBbox(rle)
        annotations.append(dict(id=char_id, bbox=bbox.astype(int).tolist(), area=int(area), segmentation=rle))
        char_id += 1
    with open(os.path.join(output_dir, f'{image_name.split(".")[0]}.json'), 'w') as f:
        json.dump(dict(image_info=image_info, annotations=annotations), f, indent=2)

In [ ]:
# 复制一下输出的label看看效果是不是对的？
# check一下area的最小值
# check到底有没有drop的

In [ ]:
import json
from pycocotools import mask as maskUtils
with open("/data/huxingjian/workspace/sam2/sav_dataset/example/sav_000001_manual.json") as f:
    data = json.load(f)
data

## SAM EVAL

### GRID MODE: GRID由于文字太稀疏导致不生效

In [24]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
import cv2
import numpy as np
import json
import os
from matplotlib import pyplot as plt
from pycocotools import mask as maskUtils

sam2_checkpoint = "sam2_logs/configs/sam2.1_training/sam_flywheel_round1/checkpoints/checkpoint_1.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"

sam2_model = build_sam2(model_cfg, sam2_checkpoint, device='cuda:1')

predictor = SAM2ImagePredictor(sam2_model)

In [25]:
def show_mask(mask, ax, random_color=False, borders = True):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30/255, 144/255, 255/255, 0.6])
    h, w = mask.shape[-2:]
    mask = mask.astype(np.uint8)
    mask_image =  mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    if borders:
        import cv2
        contours, _ = cv2.findContours(mask,cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE) 
        # Try to smooth contours
        contours = [cv2.approxPolyDP(contour, epsilon=0.01, closed=True) for contour in contours]
        mask_image = cv2.drawContours(mask_image, contours, -1, (1, 1, 1, 0.5), thickness=2) 
    ax.imshow(mask_image)

def show_points(coords, labels, ax, marker_size=375):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)   

def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor=(0, 0, 0, 0), lw=2))  
    
def show_masks(image, masks, scores, point_coords=None, box_coords=None, input_labels=None, borders=True):
    for i, (mask, score) in enumerate(zip(masks, scores)):
        plt.figure(figsize=(10, 10))
        plt.imshow(image)
        show_mask(mask, plt.gca(), borders=borders)
        if point_coords is not None:
            assert input_labels is not None
            show_points(point_coords, input_labels, plt.gca())
        if box_coords is not None:
            # boxes
            show_box(box_coords, plt.gca())
        if len(scores) > 1:
            plt.title(f"Mask {i+1}, Score: {score:.3f}", fontsize=18)
        plt.axis('off')
        plt.show()

In [ ]:
miou = []
for img_path in sorted(os.listdir('data/OBIMD_test100/JPEGImages')):
    image = cv2.imread(os.path.join('data/OBIMD_test100/JPEGImages', img_path))
    base_name = os.path.splitext(img_path)[0]
    with open(os.path.join('data/OBIMD_test100/facsimile_json', f'{base_name}.json')) as f:
        data = json.load(f)
    predictor.set_image(image)
    input_box = np.array([[0,0, image.shape[1], image.shape[0]]])
    gt_masks = cv2.imread(os.path.join('data/OBIMD_test100/VOC', f'{base_name}.png'), cv2.IMREAD_GRAYSCALE)
    masks, scores, _ = predictor.predict(
        point_coords=None,
        point_labels=None,
        box=input_box[None, :],
        multimask_output=False,
    )
    pred = 1 - masks[0] # WARING: 1 - masks[0] is used to convert the mask to binary (0 or 1) format
    gt_masks = gt_masks > 0
    miou.append(np.sum(pred * gt_masks) / (np.sum(pred) + np.sum(gt_masks) - np.sum(pred * gt_masks)))  
print(np.mean(miou))

### BOX MODE

In [5]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
import cv2
import numpy as np
import json
import os
from matplotlib import pyplot as plt
from pycocotools import mask as maskUtils

sam2_checkpoint = "sam2_logs/configs/sam2.1_training/sam2.1_hiera_l_OBIMD_stage1.yaml/checkpoints/checkpoint_10.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"

sam2_model = build_sam2(model_cfg, sam2_checkpoint, device='cuda:1')

predictor = SAM2ImagePredictor(sam2_model)

In [6]:
def show_mask(mask, ax, random_color=False, borders = True):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30/255, 144/255, 255/255, 0.6])
    h, w = mask.shape[-2:]
    mask = mask.astype(np.uint8)
    mask_image =  mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    if borders:
        import cv2
        contours, _ = cv2.findContours(mask,cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE) 
        # Try to smooth contours
        contours = [cv2.approxPolyDP(contour, epsilon=0.01, closed=True) for contour in contours]
        mask_image = cv2.drawContours(mask_image, contours, -1, (1, 1, 1, 0.5), thickness=2) 
    ax.imshow(mask_image)

def show_points(coords, labels, ax, marker_size=375):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)   

def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor=(0, 0, 0, 0), lw=2))  
    
def show_masks(image, masks, scores, point_coords=None, box_coords=None, input_labels=None, borders=True):
    for i, (mask, score) in enumerate(zip(masks, scores)):
        plt.figure(figsize=(10, 10))
        plt.imshow(image)
        show_mask(mask, plt.gca(), borders=borders)
        if point_coords is not None:
            assert input_labels is not None
            show_points(point_coords, input_labels, plt.gca())
        if box_coords is not None:
            # boxes
            show_box(box_coords, plt.gca())
        if len(scores) > 1:
            plt.title(f"Mask {i+1}, Score: {score:.3f}", fontsize=18)
        plt.axis('off')
        plt.show()

In [ ]:
miou = []
for img_path in sorted(os.listdir('data/OBIMD_test100/JPEGImages')):
    image = cv2.imread(os.path.join('data/OBIMD_test100/JPEGImages', img_path))
    base_name = os.path.splitext(img_path)[0]
    with open(os.path.join('data/OBIMD_test100/facsimile_json', f'{base_name}.json')) as f:
        data = json.load(f)
    predictor.set_image(image)
    input_box = []
    gt_masks = []
    for d in data['annotations']:
        input_box.append(d['bbox'])
        binary_mask = maskUtils.decode(d['segmentation'])
        gt_masks.append(binary_mask)
    input_box = np.array(input_box).reshape(-1, 4)
    input_box[:, 2:] += input_box[:, :2]  # Convert from [x, y, w, h] to [x0, y0, x1, y1]
    masks, scores, _ = predictor.predict(
        point_coords=None,
        point_labels=None,
        box=input_box[None, :],
        multimask_output=False,
    )
    for box, mask, gt_mask in zip(input_box, masks, gt_masks):
        x0, y0, x1, y1 = box
        pred = mask[0, y0:y1, x0:x1]
        gt = gt_mask[y0:y1, x0:x1] > 0
        if np.sum(gt) == 0:
            continue
        miou.append(np.sum(pred * gt) / (np.sum(pred) + np.sum(gt) - np.sum(pred * gt)))    
print(np.mean(miou))

## sam可视化

In [1]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
import cv2
import numpy as np
import json
import os
from matplotlib import pyplot as plt
import random

sam2_checkpoint = "/data/huxingjian/workspace/sam2/sam2_logs/configs/sam2.1_training/sam_flywheel_round2/checkpoints/checkpoint_1.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"

sam2_model = build_sam2(model_cfg, sam2_checkpoint, device='cuda:1')
predictor = SAM2ImagePredictor(sam2_model)

In [ ]:
random.seed(42)
file_list = random.sample(os.listdir("data/OBIMD_raw/rubbing"), 100) # 对齐
for img_path in sorted(file_list):
    image = cv2.imread(os.path.join('data/OBIMD_raw/rubbing', img_path))
    facsimile = cv2.imread(os.path.join('data/OBIMD_raw/facsimile', img_path))
    base_name = os.path.splitext(img_path)[0]
    with open(os.path.join('data/OBIMD_raw/facsimile_json', f'{base_name}.json')) as f:
        data = json.load(f)
    predictor.set_image(image)
    input_box = []
    for d in data['annotations']:
        input_box.append(d['bbox'])
    input_box = np.array(input_box).reshape(-1, 4)
    input_box[:, 2:] += input_box[:, :2]  # Convert from [x, y, w, h] to [x0, y0, x1, y1]
    masks, scores, _ = predictor.predict(
        point_coords=None,
        point_labels=None,
        box=input_box[None, :],
        multimask_output=False,
    )
    masks = masks.any(axis=0)[0].astype(np.uint8) * 255 # H, W?
    for box in input_box:
        x0, y0, x1, y1 = box
        image = cv2.rectangle(image, (int(x0), int(y0)), (int(x1), int(y1)), (0, 255, 0), 2)
    temp_dir = "visualization"
    os.makedirs(temp_dir, exist_ok=True)
    output_path = os.path.join(temp_dir, img_path)
    # Apply a colormap to the facsimile for better visualization
    colored_facsimile = cv2.applyColorMap(masks, cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(image, 0.5, colored_facsimile, 0.5, 0)
    combined = np.hstack((overlay, facsimile))
    cv2.imwrite(output_path, combined)

## 数据飞轮数据split

In [ ]:
import os
import re
difference = []
file = set([int(f.split('.')[0][1:]) for f in os.listdir('data/OBIMD_raw/OBIMD_rubbing') if f.startswith('h') and not f.startswith('hd')])
for unsuper_data in os.listdir("/data/huxingjian/historical_document/YQWY/HJ"):
    try:
        image_id = int(unsuper_data.split('.')[0][1:])
        if image_id not in file:
            difference.append(unsuper_data)
    except:
        # 可能是“合xxx正”这种
        image_id = int(re.findall(r'\d+', unsuper_data)[0])
        if image_id not in file:
            difference.append(unsuper_data)
        else:
            print(f"File {unsuper_data} exists in the dataset.")


## 花东数据处理

In [2]:
# 可视化
import os
import cv2
import json
from pycocotools import mask as maskUtils
import numpy as np
import random
import matplotlib.pyplot as plt
random.seed(42)
file_list = random.sample(os.listdir("data/OBIMD_stage1/facsimile_json"), 100) # fixed for same generation
for file in file_list:
    image_path = os.path.join("data/OBIMD_stage1/rubbing", file.replace('json', 'jpg'))
    facsimile = cv2.imread(os.path.join('data/OBIMD_stage1/facsimile_no_border', file.replace('json', 'jpg')))
    image = cv2.imread(image_path)
    with open(os.path.join('data/OBIMD_stage1/facsimile_json', file)) as f:
        data = json.load(f)
    mask = np.zeros_like(image[..., 0], dtype=np.uint8)
    for annotation in data['annotations']:
        bbox = annotation['bbox']
        x0, y0, w, h = bbox
        image = cv2.rectangle(image, (int(x0), int(y0)), (int(x0+w), int(y0+h)), (0, 255, 0), 2)
        try:
            iou_text = f"IoU: {annotation['predicted_iou']:.2f}"
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.5
            font_thickness = 1
            text_size = cv2.getTextSize(iou_text, font, font_scale, font_thickness)[0]
            text_x = int(x0 + (w - text_size[0]) / 2)
            text_y = int(y0 - 5)  # Position above the box
            cv2.putText(image, iou_text, (text_x, text_y), font, font_scale, (0, 255, 0), font_thickness)
        except:
            pass
        m = maskUtils.decode(annotation['segmentation'])
        mask = cv2.bitwise_or(mask, m.astype(np.uint8) * 255)

    # Convert facsimile to OpenCV format and save to temp directory
    temp_dir = "data/OBIMD_stage1/visualization"
    os.makedirs(temp_dir, exist_ok=True)
    output_path = os.path.join(temp_dir, file.replace('json', 'jpg'))
    # Apply a colormap to the facsimile for better visualization
    colored_facsimile = cv2.applyColorMap(mask, cv2.COLORMAP_JET)
    overlay = cv2.addWeighted(image, 0.5, colored_facsimile, 0.5, 0)
    combined = np.hstack((overlay, facsimile))
    cv2.imwrite(output_path, combined)

In [ ]:
# 过一遍iou的分布，看看大于0.5的有多少
import os
import json
import matplotlib.pyplot as plt
import numpy as np
ious = []
for file in os.listdir('data/OBIMD_stage2/facsimile_json'):
    with open(os.path.join('data/OBIMD_stage2/facsimile_json', file)) as f:
        data = json.load(f)
    ious.extend([annotation['predicted_iou'] for annotation in data['annotations']])
    # filter_annotions = dict(image_info=data['image_info'], annotations=[])
    # filter_annotions['annotations'] = [annotation for annotation in data['annotations'] if annotation['predicted_iou'] > 0.6]
    # if len(filter_annotions['annotations']) == 0:
    #     continue
    # with open(os.path.join('data/OBIMD_stage1/facsimile_json_iou0.6', file), 'w') as f:
    #     json.dump(filter_annotions, f, indent=2)
ious = np.array(ious)
plt.hist(ious, bins=20), len(ious[ious>0.6]), len(ious)

In [16]:
with open("data/OBIMD_raw_hj/train.txt") as f:
    train = set([l.strip() for l in f.readlines()])
with open("data/OBIMD_stage1/train.txt", 'w') as f:
    for file in os.listdir("data/OBIMD_stage1/facsimile_json_iou0.6"):
        if file.replace('json', 'jpg') in train:
            f.write(file.split('.')[0] + '\n')